<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<img src="../media/img/kisz_logo.png" width="192" height="69"> 
<h1> Working with embeddings:
<h2>An introductory workshop with applications on Semantic Search
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part 4.2 - Sentence transformers
</div>

In this notebook, we'll explore how sentence embeddings capture the meaning of entire sentences or text chunks—offering a powerful and efficient alternative to word-level embeddings like Word2Vec or even vanilla BERT.

Let's start by importing some packages.

In [6]:
import pandas as pd

# imports
from nb_config import RAW_DATA_PATH

import warnings
warnings.filterwarnings('ignore')

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Overview
</div>



**Sentence Transformers**, or SBERT, are a class of models that generate **semantically meaningful sentence embeddings**—fixed-size vector representations of entire sentences, paragraphs, or short texts. Developed by UKP Lab in 2019, SBERT builds on top of BERT and other transformer models, modifying them to efficiently handle sentence-level semantic similarity and retrieval tasks.

While models like BERT produce contextualized embeddings at the token level, they are not directly optimized for comparing entire sentences. In BERT, to compute similarity between two sentences, both need to be processed together, which is computationally expensive and impractical for large-scale tasks like semantic search. SBERT addresses this limitation by introducing a **Siamese network architecture** that allows individual sentences to be encoded independently into dense vectors, which can then be compared using simple metrics like cosine similarity.

At the heart of SBERT is a **pooling mechanism applied over the token-level outputs from a pretrained transformer** (like BERT or RoBERTa), typically using the **[CLS] token**, **mean pooling**, or **max pooling** to derive a single embedding per sentence. These sentence embeddings capture the overall meaning and context of the input, making them ideal for tasks like clustering, semantic textual similarity, paraphrase detection, and information retrieval.

SBERT is fine-tuned on sentence-pair classification tasks such as Natural Language Inference (NLI) and Semantic Textual Similarity (STS), which teaches the model to understand relationships between sentences. This fine-tuning drastically improves the quality of sentence embeddings for downstream applications.

One of the key strengths of SBERT is its efficiency. Once sentences are embedded, they can be stored and compared at scale without recomputing representations. This makes SBERT particularly powerful for real-time and large-scale applications like semantic search engines, FAQ retrieval systems, and duplicate detection in large corpora.

Since its introduction, Sentence Transformers have evolved with many lightweight and multilingual variants (e.g., all-MiniLM, distiluse, multi-qa), making them accessible for a wide range of use cases and resource-constrained environments.

Thanks to its ability to combine contextual understanding with computational efficiency, SBERT has become a foundational tool in modern natural language processing pipelines.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Preparing Sentence transformers
</div>



In [1]:
from sentence_transformers import SentenceTransformer, util

# Load a lightweight, general-purpose pretrained model
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\user\Documents\Projects\kisz-nlp-embeddings\.venv\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Tokenizing sentences and getting Embeddings
</div>

In [2]:
sentences = [
    "The weather is nice today.",
    "It's a sunny day.",
    "I love programming in Python.",
    "The stock market crashed yesterday."
]

# Generate embeddings
embeddings = model.encode(sentences, convert_to_tensor=True)


In [3]:
embeddings

tensor([[-0.0295,  0.1033,  0.1503,  ..., -0.0096, -0.1161,  0.0737],
        [-0.0434,  0.0665,  0.0798,  ...,  0.0092, -0.0820,  0.0035],
        [-0.0576,  0.0043, -0.0282,  ...,  0.1154,  0.1023, -0.0158],
        [ 0.0647, -0.0267,  0.0310,  ..., -0.0526, -0.0869,  0.0913]])

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Comparing words in different contexts
</div>

In [4]:
# Compare sentence 0 ("The weather is nice today.") with others
query_embedding = embeddings[0]
cosine_scores = util.cos_sim(query_embedding, embeddings)

# Print similarities
for i, score in enumerate(cosine_scores[0]):
    print(f"Similarity to sentence {i}: {score:.4f}")

Similarity to sentence 0: 1.0000
Similarity to sentence 1: 0.6859
Similarity to sentence 2: 0.0627
Similarity to sentence 3: 0.1769


<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Applying it to our use case
</div>

In [7]:
df = pd.read_parquet(RAW_DATA_PATH + 'movie_descriptors.parquet')

In [51]:
df.descriptor

0       Led by Woody, Andy's toys live happily in his ...
1       When siblings Judy and Peter discover an encha...
2       Obsessive master thief, Neil McCauley leads a ...
3       James Bond must unmask the mysterious head of ...
4       Widowed U.S. president Andrew Shepherd, one of...
                              ...                        
2860    An undercover MI6 agent is sent to Berlin duri...
2861    The miraculous evacuation of Allied soldiers f...
2862    When Molly Hale's sadness of her father's disa...
2863    Autobots and Decepticons are at war, with huma...
2864    An FBI agent teams with the town's veteran gam...
Name: descriptor, Length: 2865, dtype: object

embeddings = model.encode(df.descriptor, convert_to_tensor=True)

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Advantages and disadvantages of embeddings generated with Sentence transformers
</div>

Let's summarize the pros and cons of Sentence Transformers (SBERT) embeddings, and see where they can be used.

#### Advantages:

- **Semantic Sentence Embeddings**: Sentence Transformers generate fixed-size, context-aware embeddings for entire sentences or text chunks, enabling semantic comparison beyond individual words.
- **Efficient Similarity Computation**: Unlike standard BERT, SBERT is optimized for computing sentence similarity. It enables fast cosine similarity calculations between embeddings without the need to recompute representations for each pair.
- **Pretrained for Sentence-Level Tasks**: SBERT models are specifically fine-tuned on sentence-pair tasks like Natural Language Inference (NLI), making them especially effective for tasks involving semantic similarity, paraphrase detection, and retrieval.
- **Scalable for Retrieval**: SBERT can embed large text corpora in advance, allowing for efficient semantic search and ranking using vector databases (e.g., FAISS, Annoy, or Elasticsearch).
- **Plug-and-Play Usage**: SBERT is available in many variants (e.g., all-MiniLM, multi-qa, distiluse) that offer a trade-off between speed and accuracy, and can be used directly via libraries like sentence-transformers.

#### Disadvantages:

- **Fixed-Length Inputs**: Like BERT, Sentence Transformers inherit the input token limit (typically 512 tokens), which can be restrictive for longer documents or passages.
- **Lower Granularity than Word-Level Models**: SBERT produces one embedding per sentence or text chunk, which may not be ideal when word-level or fine-grained token-level representations are needed.
- **Less Effective for Out-of-Domain Data**: While powerful, SBERT models may underperform on highly domain-specific language unless fine-tuned on relevant data.
- **Preprocessing Overhead**: For tasks involving many short texts or real-time inputs, embedding and batching processes can introduce latency compared to lighter models like FastText.
- **Static Sentence Representations**: Although context-aware, SBERT embeddings are static once generated—they don't change dynamically based on task or usage context unless further fine-tuned.

#### Applications:

- **Semantic Search & Information Retrieval**: SBERT is ideal for matching queries with relevant documents, FAQs, or support tickets based on meaning rather than keywords.
- **Paraphrase Mining**: Sentence Transformers excel at identifying semantically equivalent sentences or clustering similar content, useful in deduplication or summarization pipelines.
- **Question Answering (Retrieval-Augmented)**: SBERT helps retrieve the most relevant passages or documents to answer a query, often used in hybrid QA systems before applying generative models.
- **Text Clustering & Topic Modeling**: Embeddings from SBERT can be used with clustering algorithms (e.g., KMeans, HDBSCAN) for unsupervised topic discovery or grouping similar texts.
- **Multilingual Semantic Embeddings**: With multilingual variants like distiluse-base-multilingual, SBERT supports cross-lingual tasks such as translation equivalence and multilingual search.